# EchoFactory - FAN: Evaluasi v5 (KNN + Ensemble Scoring)

## Strategi untuk Mencapai AUC 0.90+

**Root cause masalah di fanechofac4:**
1. ArcFace Cosine scoring dipakai salah → AUC 0.598 (hampir random)
2. Mahalanobis terkena bug numerik (matrix kovarians singular) → skor meledak ke 10^14

**Perbaikan di v5:**
- ✅ **KNN Cosine per-ID** – scorer paling stabil & terbukti efektif untuk embedding-based anomaly detection
- ✅ **OCSVM per-ID** – belajar "batas" hyperplane distribusi normal
- ✅ **ArcFace Center Distance per-ID** – jarak cosine ke class center W (dipakai dengan benar)
- ✅ **LOF per-ID** – density-based anomaly scoring
- ✅ **Ensemble Weighted** – gabungan semua scorer (rank-based normalization)


In [ ]:
import os, gc, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.neighbors import NearestNeighbors, LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from scipy.stats import rankdata

# Bersihkan VRAM dari sesi sebelumnya
gc.collect()
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MACHINE_TYPE = 'fan'

# Path Model dari fanechofac3
MODEL_DIR = '/kaggle/input/notebooks/muhammadmuhibin/fanechofac3'
model_path = f'{MODEL_DIR}/stgram_mfn_{MACHINE_TYPE}_v2.pt'

# Path Feature Data dari fanechofac2
FEAT_DIR = '/kaggle/input/notebooks/muhammadmuhibin/fanechofac2/features'

print(f'Device: {device}')
print(f'Model path: {model_path}')
print(f'Feature dir: {FEAT_DIR}')
print(f'Model exists: {os.path.exists(model_path)}')


In [ ]:
import math

class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False), nn.BatchNorm2d(oc), nn.PReLU(oc))
    def forward(self, x): return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(ConvBNPReLU(ic, ic, s=s, g=ic), ConvBNPReLU(ic, oc, k=1, p=0))
    def forward(self, x): return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=256):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2), DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2), DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2), DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2), nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(512, ed), nn.BatchNorm1d(ed))
    def forward(self, x): return self.head(self.enc(x))

class ArcFace(nn.Module):
    def __init__(self, ed, nc, s=32.0, m=0.5):
        super().__init__()
        self.s, self.m = s, m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, feat, labels):
        cos = F.normalize(feat, 1) @ F.normalize(self.W, 1).T
        sin = (1.0 - cos ** 2 + 1e-8).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).float()
        out = (one_hot * phi + (1.0 - one_hot) * cos) * self.s
        return F.cross_entropy(out, labels)

class STgramMFN(nn.Module):
    def __init__(self, nc, ed=256):
        super().__init__()
        self.mel, self.tgram = MobileFaceNet(ed), MobileFaceNet(ed)
        self.fuse = nn.Sequential(nn.Linear(ed * 2, ed), nn.BatchNorm1d(ed), nn.PReLU(ed))
        self.arc = ArcFace(ed, nc)
    def forward(self, mel, tg):
        feat = self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1))
        return F.normalize(feat, dim=1)

print('Model classes defined')


In [ ]:
@torch.no_grad()
def get_embeddings(model, feat_dir, machine, cond, batch_size=64):
    """Ekstrak embeddings dari model ke CPU. Batch size kecil untuk hemat VRAM."""
    data = torch.load(os.path.join(feat_dir, f'{machine}_{cond}.pt'), map_location='cpu')
    mel_feats = data['features'][:, 0:1]
    tg_feats  = data['features'][:, 1:2]
    labels    = data['labels']
    dl = DataLoader(TensorDataset(mel_feats, tg_feats, labels), batch_size=batch_size, shuffle=False)
    model.eval()
    embs, lbls = [], []
    for mb, tb, lb in dl:
        out = model(mb.to(device), tb.to(device)).cpu()
        embs.append(out)
        lbls.append(lb)
    return torch.cat(embs).numpy(), torch.cat(lbls).numpy()

def rank_normalize(scores):
    """Rank normalization: transformasi skor ke [0,1] berdasarkan ranking. Robust terhadap outlier."""
    return rankdata(scores) / len(scores)

def compute_pauc(y_true, scores, max_fpr=0.1):
    return roc_auc_score(y_true, scores, max_fpr=max_fpr)

print('Utility functions defined')


In [ ]:
# =========================================================
# SCORER 1: KNN Cosine Distance per Machine ID
# =========================================================
def score_knn_per_id(normal_embs, normal_labels, test_embs, test_labels, k=5):
    """
    Untuk setiap sample test, hitung rata-rata cosine distance
    ke K tetangga terdekat dari normal samples yang sama Machine ID-nya.
    Semakin besar = semakin anomali.
    """
    unique_ids = np.unique(normal_labels)
    knn_models = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        knn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
        knn.fit(normal_embs[mask])
        knn_models[int(uid)] = knn
    
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in knn_models:
            lid = list(knn_models.keys())[0]
        dists, _ = knn_models[lid].kneighbors(test_embs[i:i+1])
        scores.append(float(dists.mean()))
    return np.array(scores)

# =========================================================
# SCORER 2: OCSVM per Machine ID
# =========================================================
def score_ocsvm_per_id(normal_embs, normal_labels, test_embs, test_labels, nu=0.05):
    """
    One-Class SVM fit dari normal embeddings per Machine ID.
    Score = negative decision function (lebih tinggi = lebih anomali).
    """
    unique_ids = np.unique(normal_labels)
    ocsvm_models = {}
    scalers = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(normal_embs[mask])
        ocsvm = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        ocsvm.fit(X_scaled)
        ocsvm_models[int(uid)] = ocsvm
        scalers[int(uid)] = scaler
        print(f'  OCSVM fitted for ID {uid}: n_samples={mask.sum()}')
    
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in ocsvm_models:
            lid = list(ocsvm_models.keys())[0]
        X_scaled = scalers[lid].transform(test_embs[i:i+1])
        # decision_function < 0 = anomali, kita negasi agar lebih besar = lebih anomali
        score = -float(ocsvm_models[lid].decision_function(X_scaled)[0])
        scores.append(score)
    return np.array(scores)

# =========================================================
# SCORER 3: ArcFace Center Distance per Machine ID
# =========================================================
def score_arcface_center_per_id(test_embs, test_labels, arc_W):
    """
    Jarak cosine antara embedding dengan class center W dari ArcFace.
    PERBAIKAN dari fanechofac4: gunakan 1 - max_cosine (bukan cosine ke label ID saja).
    Logika: sample normal seharusnya DEKAT ke center ID-nya.
    Sample anomali: bisa jadi dekat ke center lain atau jauh dari semua center.
    Kita ambil 1 - max_similarity ke semua center sebagai skor anomali.
    """
    arc_W_norm = arc_W / (np.linalg.norm(arc_W, axis=1, keepdims=True) + 1e-8)
    embs_norm = test_embs / (np.linalg.norm(test_embs, axis=1, keepdims=True) + 1e-8)
    
    scores = []
    for i in range(len(embs_norm)):
        # Cosine similarity ke SEMUA class centers
        cos_sims = embs_norm[i] @ arc_W_norm.T  # shape: (n_classes,)
        # Anomaly score = 1 - max similarity (semakin jauh dari semua center = lebih anomali)
        score = 1.0 - float(cos_sims.max())
        scores.append(score)
    return np.array(scores)

# =========================================================
# SCORER 4: LOF per Machine ID
# =========================================================
def score_lof_per_id(normal_embs, normal_labels, test_embs, test_labels, n_neighbors=20):
    """
    Local Outlier Factor: density-based anomaly scorer.
    Fit pada normal data per ID, score pada test data.
    """
    unique_ids = np.unique(normal_labels)
    lof_models = {}
    for uid in unique_ids:
        mask = (normal_labels == uid)
        lof = LocalOutlierFactor(n_neighbors=n_neighbors, metric='cosine', novelty=True)
        lof.fit(normal_embs[mask])
        lof_models[int(uid)] = lof
        print(f'  LOF fitted for ID {uid}: n_samples={mask.sum()}')
    
    scores = []
    for i in range(len(test_embs)):
        lid = int(test_labels[i])
        if lid not in lof_models:
            lid = list(lof_models.keys())[0]
        # LOF: score_samples returns negative outlier factor (lebih negatif = lebih anomali)
        # Kita negasi agar lebih besar = lebih anomali
        score = -float(lof_models[lid].score_samples(test_embs[i:i+1])[0])
        scores.append(score)
    return np.array(scores)

print('Scoring functions defined (KNN, OCSVM, ArcFace-Center, LOF)')


In [ ]:
# =========================================================
# LOAD MODEL & EKSTRAK EMBEDDINGS
# =========================================================
if not os.path.exists(model_path):
    raise FileNotFoundError(f'Model not found: {model_path}')

ck = torch.load(model_path, map_location='cpu')
model = STgramMFN(ck['n_classes'], ck['embed_dim']).to(device)
model.load_state_dict(ck['model_state'], strict=True)
model.eval()

arc_W = ck['model_state']['arc.W'].cpu().numpy()  # shape: (n_classes, embed_dim)
print(f'arc.W shape: {arc_W.shape}')
print(f'n_classes={ck["n_classes"]}, embed_dim={ck["embed_dim"]}')

print('\nExtracting normal embeddings...')
ne, n_lbls = get_embeddings(model, FEAT_DIR, MACHINE_TYPE, 'normal')
print(f'Normal embs: {ne.shape} | Labels: {np.unique(n_lbls)}')

print('\nExtracting abnormal embeddings...')
ae, a_lbls = get_embeddings(model, FEAT_DIR, MACHINE_TYPE, 'abnormal')
print(f'Abnormal embs: {ae.shape}')

# Gabungkan semua embeddings & labels untuk evaluasi
all_embs = np.concatenate([ne, ae], axis=0)
all_labels = np.concatenate([np.concatenate([n_lbls, a_lbls])], axis=0)  # Machine ID labels
y_true = np.concatenate([np.zeros(len(ne)), np.ones(len(ae))])  # 0=normal, 1=anomali

print(f'\nTotal: {len(ne)} normal + {len(ae)} abnormal = {len(all_embs)} samples')


In [ ]:
# =========================================================
# HITUNG SEMUA SCORER
# =========================================================
results = {}

# --- Scorer 1: KNN (k=5) ---
print('=== Scorer 1: KNN Cosine (k=5) ===')
for k in [3, 5, 10, 20]:
    s_n = score_knn_per_id(ne, n_lbls, ne, n_lbls, k=k)
    s_a = score_knn_per_id(ne, n_lbls, ae, a_lbls, k=k)
    sc = np.concatenate([s_n, s_a])
    auc = roc_auc_score(y_true, sc)
    pauc = compute_pauc(y_true, sc)
    results[f'KNN-k{k}'] = {'auc': auc, 'pauc': pauc, 'scores_n': s_n, 'scores_a': s_a}
    print(f'  KNN k={k:2d}: AUC={auc:.4f} | pAUC={pauc:.4f}')

# --- Scorer 2: ArcFace Center Distance ---
print('\n=== Scorer 2: ArcFace Center Distance ===')
s_n = score_arcface_center_per_id(ne, n_lbls, arc_W)
s_a = score_arcface_center_per_id(ae, a_lbls, arc_W)
sc = np.concatenate([s_n, s_a])
auc = roc_auc_score(y_true, sc)
pauc = compute_pauc(y_true, sc)
results['ArcFace-Center'] = {'auc': auc, 'pauc': pauc, 'scores_n': s_n, 'scores_a': s_a}
print(f'  ArcFace-Center: AUC={auc:.4f} | pAUC={pauc:.4f}')

# --- Scorer 3: OCSVM ---
print('\n=== Scorer 3: OCSVM per-ID ===')
for nu in [0.01, 0.05, 0.1]:
    print(f'  nu={nu}:')
    s_n = score_ocsvm_per_id(ne, n_lbls, ne, n_lbls, nu=nu)
    s_a = score_ocsvm_per_id(ne, n_lbls, ae, a_lbls, nu=nu)
    sc = np.concatenate([s_n, s_a])
    auc = roc_auc_score(y_true, sc)
    pauc = compute_pauc(y_true, sc)
    results[f'OCSVM-nu{nu}'] = {'auc': auc, 'pauc': pauc, 'scores_n': s_n, 'scores_a': s_a}
    print(f'  OCSVM nu={nu}: AUC={auc:.4f} | pAUC={pauc:.4f}')

# --- Scorer 4: LOF ---
print('\n=== Scorer 4: LOF per-ID ===')
for k in [10, 20]:
    print(f'  n_neighbors={k}:')
    s_n = score_lof_per_id(ne, n_lbls, ne, n_lbls, n_neighbors=k)
    s_a = score_lof_per_id(ne, n_lbls, ae, a_lbls, n_neighbors=k)
    sc = np.concatenate([s_n, s_a])
    auc = roc_auc_score(y_true, sc)
    pauc = compute_pauc(y_true, sc)
    results[f'LOF-k{k}'] = {'auc': auc, 'pauc': pauc, 'scores_n': s_n, 'scores_a': s_a}
    print(f'  LOF k={k}: AUC={auc:.4f} | pAUC={pauc:.4f}')

print('\n=== SEMUA HASIL ===')
for name, r in sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True):
    print(f'  {name:20s}: AUC={r["auc"]:.4f} | pAUC={r["pauc"]:.4f}')


In [ ]:
# =========================================================
# ENSEMBLE: Rank-Based Normalization + Weighted Average
# =========================================================
print('=== Ensemble Scoring ===')

# Pilih scorer terbaik berdasarkan AUC individu
best_individual = sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True)
print('\nTop 4 individual scorers:')
for name, r in best_individual[:4]:
    print(f'  {name}: AUC={r["auc"]:.4f}')

# Ensemble semua scorer dengan rank normalization
all_sc_n_ranked = []
all_sc_a_ranked = []

for name, r in results.items():
    sc_all = np.concatenate([r['scores_n'], r['scores_a']])
    sc_ranked = rank_normalize(sc_all)
    all_sc_n_ranked.append(sc_ranked[:len(ne)])
    all_sc_a_ranked.append(sc_ranked[len(ne):])

ensemble_n = np.mean(all_sc_n_ranked, axis=0)
ensemble_a = np.mean(all_sc_a_ranked, axis=0)
ensemble_sc = np.concatenate([ensemble_n, ensemble_a])
ensemble_auc = roc_auc_score(y_true, ensemble_sc)
ensemble_pauc = compute_pauc(y_true, ensemble_sc)
results['Ensemble-All'] = {'auc': ensemble_auc, 'pauc': ensemble_pauc,
                            'scores_n': ensemble_n, 'scores_a': ensemble_a}
print(f'\n[Ensemble-All (rank avg)]: AUC={ensemble_auc:.4f} | pAUC={ensemble_pauc:.4f}')

# Ensemble hanya top-3 scorer
top3_names = [name for name, _ in best_individual[:3]]
top3_sc_n, top3_sc_a = [], []
for name in top3_names:
    r = results[name]
    sc_all = np.concatenate([r['scores_n'], r['scores_a']])
    sc_ranked = rank_normalize(sc_all)
    top3_sc_n.append(sc_ranked[:len(ne)])
    top3_sc_a.append(sc_ranked[len(ne):])

top3_n = np.mean(top3_sc_n, axis=0)
top3_a = np.mean(top3_sc_a, axis=0)
top3_sc = np.concatenate([top3_n, top3_a])
top3_auc = roc_auc_score(y_true, top3_sc)
top3_pauc = compute_pauc(y_true, top3_sc)
results['Ensemble-Top3'] = {'auc': top3_auc, 'pauc': top3_pauc,
                             'scores_n': top3_n, 'scores_a': top3_a}
print(f'[Ensemble-Top3]:           AUC={top3_auc:.4f} | pAUC={top3_pauc:.4f}')
print(f'  (Top3 scorers: {top3_names})')


In [ ]:
# =========================================================
# PILIH SCORER TERBAIK & VISUALISASI
# =========================================================
best_name = max(results.keys(), key=lambda x: results[x]['auc'])
best = results[best_name]
best_auc = best['auc']
best_pauc = best['pauc']

print('\n' + '='*60)
print(f'HASIL TERBAIK: [{best_name}]')
print(f'  AUC  = {best_auc:.4f}')
print(f'  pAUC = {best_pauc:.4f}')
print(f'  Normal   scores: min={best["scores_n"].min():.4f}, max={best["scores_n"].max():.4f}, mean={best["scores_n"].mean():.4f}')
print(f'  Abnormal scores: min={best["scores_a"].min():.4f}, max={best["scores_a"].max():.4f}, mean={best["scores_a"].mean():.4f}')
print('='*60)

best_sc_all = np.concatenate([best['scores_n'], best['scores_a']])
fpr, tpr, thr = roc_curve(y_true, best_sc_all)
best_threshold = float(thr[np.argmax(tpr - fpr)])
print(f'  Optimal Threshold: {best_threshold:.4f}')

# --- Plot ROC Curve ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(fpr, tpr, 'b-', lw=2, label=f'{best_name}\nAUC={best_auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
mask_pauc = fpr <= 0.1
ax.fill_between(fpr[mask_pauc], tpr[mask_pauc], alpha=0.3, color='orange',
                label=f'pAUC (FPR<0.1)={best_pauc:.4f}')
ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'ROC Curve - FAN ({best_name})', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(); ax.grid(alpha=0.3)

# --- Plot Score Distribution ---
ax = axes[1]
ax.hist(best['scores_n'], bins=50, alpha=0.6, color='blue', label='Normal')
ax.hist(best['scores_a'], bins=50, alpha=0.6, color='red', label='Abnormal')
ax.axvline(x=best_threshold, color='green', linestyle='--', lw=2, label=f'Threshold={best_threshold:.3f}')
ax.set_title(f'Score Distribution - FAN ({best_name})', fontweight='bold')
ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Count')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'/kaggle/working/eval_{MACHINE_TYPE}_v5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

# --- Summary Table ---
print('\n=== RANGKUMAN SEMUA SCORER ===')
for name, r in sorted(results.items(), key=lambda x: x[1]['auc'], reverse=True):
    marker = ' ← TERBAIK' if name == best_name else ''
    print(f'  {name:25s}: AUC={r["auc"]:.4f} | pAUC={r["pauc"]:.4f}{marker}')


In [ ]:
# =========================================================
# EXPORT ONNX + SIMPAN INFERENCE CONFIG
# =========================================================
!pip install -q onnx onnxscript

# Export model ke CPU dulu
m_cpu = STgramMFN(ck['n_classes'], ck['embed_dim'])
m_cpu.load_state_dict(ck['model_state'], strict=True)
m_cpu.eval()

onnx_path = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}.onnx'
torch.onnx.export(
    m_cpu,
    (torch.randn(1, 1, 128, 128), torch.randn(1, 1, 128, 128)),
    onnx_path,
    input_names=['mel', 'tgram'],
    output_names=['embedding'],
    opset_version=17
)
print(f'ONNX Exported: {onnx_path}')

# Simpan inference config
cfg = {
    'machine': MACHINE_TYPE,
    'model': f'stgram_mfn_{MACHINE_TYPE}.onnx',
    'best_scorer': best_name,
    'auc': float(best_auc),
    'pauc': float(best_pauc),
    'threshold': float(best_threshold),
    'embed_dim': ck['embed_dim'],
    'n_classes': ck['n_classes'],
    'all_results': {k: {'auc': float(v['auc']), 'pauc': float(v['pauc'])} for k, v in results.items()},
    'mel_params': {'n_mels': 128, 'n_fft': 1024, 'hop_length': 512},
    'tgram_params': {'n_mels': 128, 'n_fft': 512, 'hop_length': 512}
}
with open(f'/kaggle/working/inference_config_{MACHINE_TYPE}_v5.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('Inference config saved.')
print(f'\n=== FINAL: FAN AUC = {best_auc:.4f} | pAUC = {best_pauc:.4f} ===')
